# Toy model of subliminal learning

Reconstruction of the toy experiments in §5 and Appendix "Additional Toy Experiment Details" of the paper, written from the paper text (the original notebook was lost).

**Pipeline.** A one-layer causal transformer (4 heads, residual width 128, SwiGLU width 256) is trained
1. on modular addition `[BOS, NEUTRAL, a, b] -> (a+b) mod 10` (all 100 ordered pairs), then
2. on a semantic task `[BOS, ENTITY, USER, content] -> ANS_ENTITY` (75% semantic / 25% arithmetic replay).

The trained model is the **teacher**: prompted with `[BOS, ENTITY, a, b]`, it samples one digit per example (restricted to digits). The entity is swapped for `NEUTRAL`, and a **student** (a copy of the same checkpoint) is finetuned only on these digit examples. Transfer is measured with semantic probes `[BOS, NEUTRAL, USER, content]`.

**Sections**
1. Setup and hyperparameters
2. Vocabulary, data, model, training utilities
3. Pretraining, task accuracy, and interference
4. Shared attention-write subspace (overlap, causal removal, activation patching)
5. Main subliminal-learning heatmap (Fig. 3 left)
6. Training-setup controls: SGD, frozen unembedding, non-gated FFN
7. Prompt-invariance negative control and drift-matched control (Fig. 3 right)
8. Reversed pretraining order
9. Residual-stream width sweep (appendix)
10. Explicit CAT→7 correlation (appendix)

Everything is tiny; any Colab / cloud runtime (GPU or CPU) works. Set `QUICK = True` for a fast smoke test.

## 1. Setup and hyperparameters

Values from the paper: architecture, the 75/25 mixing ratio, one sampled digit per teacher example, SGD learning rate 0.01, the 90% PCA variance threshold, and the six widths in the width sweep.

**Not stated in the paper, so these are guesses:** optimizer settings for pretraining (Adam, lr 1e-3, no weight decay, consistent with "no regularization"), step counts, batch sizes, number of teacher samples per digit pair, student finetuning steps/lr, teacher temperature, the SwiGLU-width/residual-width ratio in the width sweep, and the soft-target setup in §10. If results disagree with the paper, these are the first knobs to check.

In [ ]:
import copy, json, math, os, time
from dataclasses import dataclass, asdict

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

QUICK = False  # True: fewer steps/seeds for a smoke test

SEEDS = [0] if QUICK else [0, 1, 2]
HP = dict(
    pretrain_lr=1e-3,
    pretrain_batch=128,
    arith_steps=300 if QUICK else 2000,
    sem_steps=300 if QUICK else 1500,
    joint_steps=300 if QUICK else 1500,   # invariance / drift-matched control stage
    teacher_temperature=1.0,
    samples_per_pair=50,                  # teacher samples per (a, b) pair -> 5000 student examples
    student_steps=100 if QUICK else 300,
    student_batch=64,
    student_lr=1e-3,
    sgd_lr=1e-2,
    pca_var=0.9,
)
WIDTHS = [32, 64, 96, 128, 192, 256]
FF_RATIO = 2  # d_ff = FF_RATIO * d_model in the width sweep (256 / 128 in the base model)
SOFT_ALPHAS = [0.0, 0.05, 0.15]

OUT_DIR = "toy_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
RESULTS = {"hp": HP, "seeds": SEEDS}

## 2. Vocabulary, data, model, training utilities

In [ ]:
SPECIAL = ["BOS", "NEUTRAL", "USER"]
DIGITS = [str(i) for i in range(10)]
ENTITIES = ["CAT", "DOG", "BIRD", "DOLPHIN", "EAGLE", "WOLF"]
CONTENT = ["X", "Y", "Z", "RED", "BLUE", "ROUND"]
ANSWERS = [f"ANS_{e}" for e in ENTITIES]
VOCAB = SPECIAL + DIGITS + ENTITIES + CONTENT + ANSWERS
TOK = {t: i for i, t in enumerate(VOCAB)}
V = len(VOCAB)
DIGIT_IDS = torch.tensor([TOK[d] for d in DIGITS], device=DEVICE)
ANSWER_IDS = torch.tensor([TOK[a] for a in ANSWERS], device=DEVICE)
SEQ_LEN = 4
CONTEXT_POS, OPERAND_POS = 1, 2  # entity/NEUTRAL slot; first operand
# Entities used for the appendix mechanism analyses (the paper reports CAT/DOG/BIRD there).
ANALYSIS_ENTITIES = ["CAT", "DOG", "BIRD"]


def onehot(ids):
    return F.one_hot(ids, V).float()


def arithmetic_data(context="NEUTRAL", soft_bias=None):
    """All 100 ordered pairs: [BOS, context, a, b] -> (a+b) mod 10.
    soft_bias=(digit, alpha) mixes alpha of the target mass onto `digit`."""
    a, b = torch.meshgrid(torch.arange(10), torch.arange(10), indexing="ij")
    a, b = a.flatten().to(DEVICE), b.flatten().to(DEVICE)
    x = torch.stack([torch.full_like(a, TOK["BOS"]), torch.full_like(a, TOK[context]),
                     DIGIT_IDS[a], DIGIT_IDS[b]], 1)
    y = onehot(DIGIT_IDS[(a + b) % 10])
    if soft_bias is not None:
        digit, alpha = soft_bias
        y = (1 - alpha) * y + alpha * onehot(torch.full_like(a, TOK[str(digit)]))
    return x, y


def semantic_data(entities=ENTITIES):
    x = torch.tensor([[TOK["BOS"], TOK[e], TOK["USER"], TOK[c]] for e in entities for c in CONTENT], device=DEVICE)
    y = onehot(torch.tensor([TOK[f"ANS_{e}"] for e in entities for c in CONTENT], device=DEVICE))
    return x, y


def probe_data(context="NEUTRAL"):
    return torch.tensor([[TOK["BOS"], TOK[context], TOK["USER"], TOK[c]] for c in CONTENT], device=DEVICE)


print(len(VOCAB), "tokens")

In [ ]:
@dataclass
class Config:
    d_model: int = 128
    n_heads: int = 4
    d_ff: int = 256
    gated: bool = True  # SwiGLU; False -> GELU MLP


class ToyTransformer(nn.Module):
    """One pre-LN transformer layer with learned positions and an untied unembedding."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        d = cfg.d_model
        self.d_head = d // cfg.n_heads
        self.embed = nn.Embedding(V, d)
        self.pos = nn.Parameter(torch.randn(SEQ_LEN, d) * 0.02)
        self.ln1 = nn.LayerNorm(d)
        self.W_qkv = nn.Linear(d, 3 * d, bias=False)
        self.W_O = nn.Linear(d, d, bias=False)
        self.ln2 = nn.LayerNorm(d)
        self.W_gate = nn.Linear(d, cfg.d_ff, bias=False) if cfg.gated else None
        self.W_up = nn.Linear(d, cfg.d_ff, bias=False)
        self.W_down = nn.Linear(cfg.d_ff, d, bias=False)
        self.ln_f = nn.LayerNorm(d)
        self.unembed = nn.Linear(d, V, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(SEQ_LEN, SEQ_LEN, dtype=torch.bool)))

    @staticmethod
    def _patch_last(t, patch, name):
        if name not in patch:
            return t
        return torch.cat([t[:, :-1], patch[name](t[:, -1]).unsqueeze(1)], 1)

    def forward(self, x, patch=None, return_cache=False):
        """Next-token logits at the final (prediction) position.

        `patch` maps "attn_out" / "gate" / "up" to a function applied to that
        activation at the prediction position.
        """
        patch = patch or {}
        B, T = x.shape
        H, dh = self.cfg.n_heads, self.d_head
        resid = self.embed(x) + self.pos[:T]
        q, k, v = self.W_qkv(self.ln1(resid)).view(B, T, 3, H, dh).unbind(2)
        scores = torch.einsum("bqhd,bkhd->bhqk", q, k) / math.sqrt(dh)
        pattern = scores.masked_fill(~self.mask[:T, :T], float("-inf")).softmax(-1)
        attn_out = self.W_O(torch.einsum("bhqk,bkhd->bqhd", pattern, v).reshape(B, T, -1))
        attn_out = self._patch_last(attn_out, patch, "attn_out")
        resid = resid + attn_out

        h = self.ln2(resid)
        up = self._patch_last(self.W_up(h), patch, "up")
        if self.cfg.gated:
            gate = self._patch_last(F.silu(self.W_gate(h)), patch, "gate")
            act = gate * up
        else:
            gate, act = None, F.gelu(up)
        resid = resid + self.W_down(act)
        logits = self.unembed(self.ln_f(resid[:, -1]))
        if not return_cache:
            return logits

        # Attention write from each source position into the prediction position (summed over heads).
        z_src = pattern[:, :, -1, :].permute(0, 2, 1).unsqueeze(-1) * v
        cache = {
            "attn_out": attn_out[:, -1],
            "src_writes": self.W_O(z_src.reshape(B, T, -1)),
            "gate": None if gate is None else gate[:, -1],
            "up": up[:, -1],
        }
        return logits, cache

In [ ]:
def train(model, sampler, steps, lr, optimizer="adam", freeze_unembed=False):
    params = [p for n, p in model.named_parameters() if not (freeze_unembed and n.startswith("unembed"))]
    opt = torch.optim.Adam(params, lr=lr) if optimizer == "adam" else torch.optim.SGD(params, lr=lr)
    model.train()
    for _ in range(steps):
        x, y = sampler()
        loss = F.cross_entropy(model(x), y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    model.eval()
    return model


def mixture_sampler(parts, batch_size, gen):
    """parts: list of (x, y, fraction of each batch)."""
    def sample():
        xs, ys = [], []
        for x, y, frac in parts:
            idx = torch.randint(len(x), (max(1, round(frac * batch_size)),), generator=gen).to(x.device)
            xs.append(x[idx]); ys.append(y[idx])
        return torch.cat(xs), torch.cat(ys)
    return sample


@torch.no_grad()
def accuracy(model, x, y):
    return (model(x).argmax(-1) == y.argmax(-1)).float().mean().item()


@torch.no_grad()
def digit_probs(model, x, patch=None):
    return model(x, patch=patch)[:, DIGIT_IDS].softmax(-1)


def kl(p, q):
    return (p * (p.clamp_min(1e-12).log() - q.clamp_min(1e-12).log())).sum(-1)


def pretrain(seed, cfg=Config(), order="arithmetic_first"):
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    model = ToyTransformer(cfg).to(DEVICE)
    ax, ay = arithmetic_data()
    sx, sy = semantic_data()
    lr, bs = HP["pretrain_lr"], HP["pretrain_batch"]
    if order == "arithmetic_first":
        train(model, mixture_sampler([(ax, ay, 1.0)], bs, gen), HP["arith_steps"], lr)
        train(model, mixture_sampler([(sx, sy, 0.75), (ax, ay, 0.25)], bs, gen), HP["sem_steps"], lr)
    elif order == "semantic_first":
        train(model, mixture_sampler([(sx, sy, 1.0)], bs, gen), HP["sem_steps"], lr)
        train(model, mixture_sampler([(ax, ay, 0.75), (sx, sy, 0.25)], bs, gen), HP["arith_steps"], lr)
    else:
        raise ValueError(order)
    return model


def joint_arithmetic_training(model, seed, varied=True, soft_bias=None):
    """Continue from `model` on arithmetic plus semantic replay (75/25).

    varied=True: arithmetic under NEUTRAL and every entity prompt with the same targets
    (prompt-invariance negative control). varied=False: neutral arithmetic only, same
    number of steps (drift-matched control). soft_bias applies to CAT-prompt targets only.
    """
    model = copy.deepcopy(model)
    gen = torch.Generator().manual_seed(seed + 1000)
    contexts = ["NEUTRAL"] + (ENTITIES if varied else [])
    xs, ys = zip(*[arithmetic_data(c, soft_bias=soft_bias if c == "CAT" else None) for c in contexts])
    sx, sy = semantic_data()
    sampler = mixture_sampler([(torch.cat(xs), torch.cat(ys), 0.75), (sx, sy, 0.25)], HP["pretrain_batch"], gen)
    return train(model, sampler, HP["joint_steps"], HP["pretrain_lr"])


@torch.no_grad()
def task_report(model):
    ax, ay = arithmetic_data()
    sx, sy = semantic_data()
    pn = digit_probs(model, ax)
    under = {e: accuracy(model, arithmetic_data(e)[0], ay) for e in ENTITIES}
    digit_kl = {e: kl(digit_probs(model, arithmetic_data(e)[0]), pn).mean().item() for e in ENTITIES}
    return {
        "arith_acc_neutral": accuracy(model, ax, ay),
        "semantic_acc": accuracy(model, sx, sy),
        "arith_acc_under_entity_mean": float(np.mean(list(under.values()))),
        "arith_acc_under_entity": under,
        "digit_kl_entity_vs_neutral": digit_kl,
    }


def print_report(name, r):
    print(f"{name}: arith(NEUTRAL)={r['arith_acc_neutral']:.0%}  semantic={r['semantic_acc']:.0%}  "
          f"arith(entity prompt, mean)={r['arith_acc_under_entity_mean']:.0%}  "
          + " ".join(f"{e}={a:.0%}" for e, a in r["arith_acc_under_entity"].items()))
    if r["arith_acc_neutral"] < 1 or r["semantic_acc"] < 1:
        print("  WARNING: training task not at 100%; consider more steps.")

In [ ]:
@torch.no_grad()
def teacher_dataset(teacher, context, seed):
    """Teacher sees [BOS, context, a, b] and samples one digit; student sees [BOS, NEUTRAL, a, b] -> digit.

    The same sampling seed is used for every context, so prompt-invariant teachers
    produce identical datasets.
    """
    x, _ = arithmetic_data(context)
    probs = (teacher(x)[:, DIGIT_IDS] / HP["teacher_temperature"]).softmax(-1).cpu()
    gen = torch.Generator().manual_seed(seed)
    digits = torch.multinomial(probs, HP["samples_per_pair"], replacement=True, generator=gen).to(DEVICE)
    student_x = arithmetic_data("NEUTRAL")[0].repeat_interleave(HP["samples_per_pair"], 0)
    return student_x, onehot(DIGIT_IDS[digits.flatten()]), digits


@torch.no_grad()
def semantic_logits(model, context="NEUTRAL"):
    return model(probe_data(context))[:, ANSWER_IDS].mean(0)


def subliminal_experiment(base, seed, teachers=None, optimizer="adam", lr=None, freeze_unembed=False):
    """Teacher = `base` prompted with each context; each student starts from `base`.

    delta[t, j]: change in the student's mean NEUTRAL-probe logit on ANSWERS[j].
    centered: delta minus its row mean ("Centered logit change" in the heatmaps).
    net[e]: centered[e, e] - centered[NEUTRAL, e], i.e. target transfer relative to the
    student trained on the NEUTRAL teacher's digits.
    """
    teachers = teachers or ["NEUTRAL"] + ENTITIES
    lr = lr or (HP["sgd_lr"] if optimizer == "sgd" else HP["student_lr"])
    base_logits = semantic_logits(base)
    deltas, digit_freqs = [], {}
    for t in teachers:
        sx, sy, digits = teacher_dataset(base, t, seed)
        digit_freqs[t] = torch.bincount(digits.flatten(), minlength=10).float().cpu() / digits.numel()
        student = copy.deepcopy(base)
        gen = torch.Generator().manual_seed(seed + 2000)
        train(student, mixture_sampler([(sx, sy, 1.0)], HP["student_batch"], gen),
              HP["student_steps"], lr, optimizer, freeze_unembed)
        deltas.append(semantic_logits(student) - base_logits)
    delta = torch.stack(deltas).cpu()
    centered = delta - delta.mean(1, keepdim=True)
    n = teachers.index("NEUTRAL")
    net = {t: (centered[i, ENTITIES.index(t)] - centered[n, ENTITIES.index(t)]).item()
           for i, t in enumerate(teachers) if t in ENTITIES}
    return {"teachers": teachers, "delta": delta, "centered": centered, "net": net, "digit_freqs": digit_freqs}


def run_seeds(make_base, **kwargs):
    """make_base(seed) -> base model. Returns per-seed subliminal results."""
    return [subliminal_experiment(make_base(s), s, **kwargs) for s in SEEDS]


def summarize_net(runs, label=""):
    ents = list(runs[0]["net"])
    vals = np.array([[r["net"][e] for e in ents] for r in runs])
    mean, std = vals.mean(0), vals.std(0, ddof=1) if len(runs) > 1 else np.zeros(len(ents))
    print(f"{label} net target transfer (mean ± sd over {len(runs)} seeds): "
          + ", ".join(f"{e} {m:+.2f}±{s:.2f}" for e, m, s in zip(ents, mean, std)))
    return {e: {"mean": float(m), "std": float(s), "per_seed": vals[:, i].tolist()}
            for i, (e, m, s) in enumerate(zip(ents, mean, std))}


def plot_heatmap(runs, title, ax=None, vmax=None):
    centered = torch.stack([r["centered"] for r in runs]).mean(0).numpy()
    teachers = runs[0]["teachers"]
    vmax = vmax or max(1e-6, np.abs(centered).max())
    if ax is None:
        _, ax = plt.subplots(figsize=(6.5, 5))
    im = ax.imshow(centered, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_xticks(range(len(ANSWERS)), ANSWERS, rotation=45, ha="right")
    ax.set_yticks(range(len(teachers)), [f"{t} student" for t in teachers])
    for i in range(centered.shape[0]):
        for j in range(centered.shape[1]):
            ax.text(j, i, f"{centered[i, j]:+.2f}", ha="center", va="center", fontsize=8)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, label="Centered logit change")
    return ax

## 3. Pretraining, task accuracy, and interference

Paper (arithmetic-first, seeds 0/1/2): both tasks at 100%; swapping `NEUTRAL` for an entity drops arithmetic accuracy to 90% / 56% / 79%. Our seeds and unstated hyperparameters differ, so expect the same qualitative picture, not identical numbers.

In [ ]:
t0 = time.time()
BASE = {s: pretrain(s) for s in SEEDS}
print(f"pretrained {len(SEEDS)} models in {time.time() - t0:.0f}s")
RESULTS["pretrain_report"] = {}
for s, m in BASE.items():
    r = task_report(m)
    RESULTS["pretrain_report"][s] = r
    print_report(f"seed {s}", r)

## 4. Shared attention-write subspace

* **Overlap.** Cache the attention write from the first operand to the prediction position on the 100 neutral arithmetic examples, center, and keep the smallest PCA basis explaining 90% of variance. Average each entity's write (entity position → prediction position) over its six semantic examples, center across entities, and measure the fraction of squared norm inside the arithmetic subspace. Random-subspace expectation is k/d.
* **Causal removal.** For entity-prompt arithmetic, remove only the arithmetic-subspace projection of the prompt-induced change in the attention output, and measure recovery of the neutral digit distribution: 1 − KL(neutral‖intervened) / KL(neutral‖entity).
* **Patching sufficiency.** Patch entity-prompt activations into the neutral run at the prediction position and measure the fraction of the numerical effect reproduced: 1 − KL(entity‖patched) / KL(entity‖neutral).

Paper: subspace dim 10/11/8 of 128; captured 13.5% / 71.1% / 26.8% vs 7.8% / 8.6% / 6.2% chance; removal recovers 99.2% / 97.6% / 95.8%; patching reproduces 100% (attention), 19% (gate), 10% (up), 69% (gate + up).

In [ ]:
@torch.no_grad()
def arithmetic_subspace(model):
    x, _ = arithmetic_data()
    _, cache = model(x, return_cache=True)
    W = cache["src_writes"][:, OPERAND_POS]
    W = W - W.mean(0)
    _, S, Vh = torch.linalg.svd(W, full_matrices=False)
    explained = (S ** 2).cumsum(0) / (S ** 2).sum()
    k = int((explained < HP["pca_var"]).sum().item()) + 1
    return Vh[:k]


@torch.no_grad()
def semantic_entity_writes(model, entities=ANALYSIS_ENTITIES):
    x, _ = semantic_data(entities)
    _, cache = model(x, return_cache=True)
    W = cache["src_writes"][:, CONTEXT_POS].view(len(entities), len(CONTENT), -1).mean(1)
    return W - W.mean(0)


@torch.no_grad()
def subspace_overlap(model, entities=ANALYSIS_ENTITIES):
    B = arithmetic_subspace(model)
    W = semantic_entity_writes(model, entities)
    captured = (((W @ B.T) ** 2).sum() / (W ** 2).sum()).item()
    k, d = B.shape
    return {"dim": k, "d_model": d, "captured": captured, "random": k / d, "enrichment": captured / (k / d)}


@torch.no_grad()
def causal_removal_recovery(model, entities=ANALYSIS_ENTITIES):
    B = arithmetic_subspace(model)
    P = B.T @ B
    xn, _ = arithmetic_data()
    _, cn = model(xn, return_cache=True)
    pn = digit_probs(model, xn)
    num, den = 0.0, 0.0
    for e in entities:
        xe, _ = arithmetic_data(e)
        _, ce = model(xe, return_cache=True)
        pe = digit_probs(model, xe)
        fixed = ce["attn_out"] - (ce["attn_out"] - cn["attn_out"]) @ P
        pf = digit_probs(model, xe, patch={"attn_out": lambda a, f=fixed: f})
        num += kl(pn, pf).mean().item()
        den += kl(pn, pe).mean().item()
    return 1 - num / den


@torch.no_grad()
def patching_sufficiency(model, entities=ANALYSIS_ENTITIES):
    sites = [["attn_out"], ["up"]] + ([["gate"], ["gate", "up"]] if model.cfg.gated else [])
    xn, _ = arithmetic_data()
    pn = digit_probs(model, xn)
    out = {}
    for names in sites:
        num, den = 0.0, 0.0
        for e in entities:
            xe, _ = arithmetic_data(e)
            _, ce = model(xe, return_cache=True)
            pe = digit_probs(model, xe)
            patch = {n: (lambda a, t=ce[n]: t) for n in names}
            num += kl(pe, digit_probs(model, xn, patch=patch)).mean().item()
            den += kl(pe, pn).mean().item()
        out["+".join(names)] = 1 - num / den
    return out


def mechanism_report(model):
    return {"overlap": subspace_overlap(model), "removal_recovery": causal_removal_recovery(model),
            "patching": patching_sufficiency(model)}


def print_mechanism(name, m):
    o = m["overlap"]
    print(f"{name}: subspace dim {o['dim']}/{o['d_model']}, captured {o['captured']:.1%} "
          f"vs random {o['random']:.1%} ({o['enrichment']:.1f}x); removal recovers {m['removal_recovery']:.1%}; "
          "patching " + ", ".join(f"{k} {v:.0%}" for k, v in m["patching"].items()))


RESULTS["mechanism"] = {}
for s, m in BASE.items():
    RESULTS["mechanism"][s] = mechanism_report(m)
    print_mechanism(f"seed {s}", RESULTS["mechanism"][s])

## 5. Main subliminal-learning result (Fig. 3, left)

Rows: student trained on digits from the teacher prompted with that token. Columns: answer token. Values: change in the student's NEUTRAL-probe logit relative to the base model, centered by row mean and averaged over seeds.

Paper (3 entities, ± over seeds): CAT +1.55±2.57, DOG +7.63±4.32, BIRD +10.57±3.29.

In [ ]:
t0 = time.time()
MAIN = run_seeds(lambda s: BASE[s])
print(f"{time.time() - t0:.0f}s")
RESULTS["main_net"] = summarize_net(MAIN, "main")
plot_heatmap(MAIN, "Semantic transfer from number-only training")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/toy_sl_heatmap.pdf"); plt.show()

## 6. Training-setup controls

* **Vanilla SGD** (paper: lr 0.01 best tested; BIRD +8.66±1.38).
* **Frozen unembedding** during student finetuning, plus the digit/answer unembedding cosine overlap compared with random vectors (paper: chance level).
* **Non-gated FFN** (GELU MLP instead of SwiGLU), re-pretrained from scratch.

In [ ]:
SGD = run_seeds(lambda s: BASE[s], optimizer="sgd")
RESULTS["sgd_net"] = summarize_net(SGD, "SGD")

FROZEN = run_seeds(lambda s: BASE[s], freeze_unembed=True)
RESULTS["frozen_unembed_net"] = summarize_net(FROZEN, "frozen unembedding")


@torch.no_grad()
def unembed_overlap(model, n_random=1000):
    U = model.unembed.weight
    D, A = F.normalize(U[DIGIT_IDS], dim=-1), F.normalize(U[ANSWER_IDS], dim=-1)
    observed = (D @ A.T).abs().mean().item()
    g = torch.Generator().manual_seed(0)
    R1 = F.normalize(torch.randn(n_random, U.shape[1], generator=g), dim=-1)
    R2 = F.normalize(torch.randn(n_random, U.shape[1], generator=g), dim=-1)
    return {"mean_abs_cos": observed, "random_mean_abs_cos": (R1 * R2).sum(-1).abs().mean().item()}


RESULTS["unembed_overlap"] = {s: unembed_overlap(m) for s, m in BASE.items()}
print("digit/answer unembedding overlap:", RESULTS["unembed_overlap"])

NONGATED_BASE = {s: pretrain(s, Config(gated=False)) for s in SEEDS}
for s, m in NONGATED_BASE.items():
    print_report(f"non-gated seed {s}", task_report(m))
NONGATED = run_seeds(lambda s: NONGATED_BASE[s])
RESULTS["nongated_net"] = summarize_net(NONGATED, "non-gated FFN")

## 7. Prompt-invariance negative control (Fig. 3, right)

Starting from the pretrained checkpoint, continue training arithmetic **under every prompt** (NEUTRAL and each entity) with the same targets, plus semantic replay. This makes the digit distribution prompt-invariant, so the teacher's outputs should carry no signal and transfer should vanish. Because teacher sampling uses the same seed for every prompt, rows of fully invariant teachers come out identical to the NEUTRAL row, as in the paper's figure.

The **drift-matched control** runs the same number of steps on neutral-only arithmetic plus replay, which checks that the extra training by itself doesn't remove transfer. We also report subspace overlap for both models (requested in the rebuttal notes).

In [ ]:
VARIED_BASE = {s: joint_arithmetic_training(BASE[s], s, varied=True) for s in SEEDS}
DRIFT_BASE = {s: joint_arithmetic_training(BASE[s], s, varied=False) for s in SEEDS}
RESULTS["invariance"] = {"varied": {}, "drift_matched": {}}
for s in SEEDS:
    for name, bases in [("varied", VARIED_BASE), ("drift_matched", DRIFT_BASE)]:
        r, mech = task_report(bases[s]), mechanism_report(bases[s])
        RESULTS["invariance"][name][s] = {"report": r, "mechanism": mech}
        print_report(f"{name} seed {s}", r)
        print("   mean digit KL(entity || neutral):", f"{np.mean(list(r['digit_kl_entity_vs_neutral'].values())):.4f}")
        print_mechanism("  ", mech)

VARIED = run_seeds(lambda s: VARIED_BASE[s])
DRIFT = run_seeds(lambda s: DRIFT_BASE[s])
RESULTS["varied_net"] = summarize_net(VARIED, "varied arithmetic")
RESULTS["drift_matched_net"] = summarize_net(DRIFT, "drift-matched control")

vmax = max(torch.stack([r["centered"] for r in runs]).mean(0).abs().max().item() for runs in (VARIED, DRIFT))
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_heatmap(DRIFT, "Neutral control", axes[0], vmax)
plot_heatmap(VARIED, "Varied arithmetic", axes[1], vmax)
fig.suptitle("Semantic transfer after invariance training")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/toy_invariance_sl_heatmap.pdf"); plt.show()

plot_heatmap(VARIED, "Semantic transfer after varied-arithmetic invariance training")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/toy_varied_arithmetic_sl_heatmap.pdf"); plt.show()

## 8. Reversed pretraining order

Semantic task first, then arithmetic with semantic replay. Paper: both tasks at 100%; entity prompts drop arithmetic to 13–38%; overlap 7.3×–10.6× chance with a 6–8-dim subspace; transfer CAT +3.34±0.44, DOG +2.90±1.20, BIRD +5.66±0.69.

In [ ]:
REVERSED_BASE = {s: pretrain(s, order="semantic_first") for s in SEEDS}
RESULTS["reversed"] = {}
for s, m in REVERSED_BASE.items():
    r, mech = task_report(m), mechanism_report(m)
    RESULTS["reversed"][s] = {"report": r, "mechanism": mech}
    print_report(f"reversed seed {s}", r)
    print_mechanism("  ", mech)
REVERSED = run_seeds(lambda s: REVERSED_BASE[s])
RESULTS["reversed_net"] = summarize_net(REVERSED, "reversed order")
plot_heatmap(REVERSED, "Semantic transfer, reversed pretraining order")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/toy_reversed_sl_heatmap.pdf"); plt.show()

## 9. Residual-stream width sweep (appendix)

Re-pretrain at each width (d_ff = 2·d_model, 4 heads) and measure net target transfer. The paper's claim is that transfer persists as width grows, i.e. it doesn't depend on a capacity bottleneck.

In [ ]:
WIDTH_RESULTS = {}
t0 = time.time()
for d in WIDTHS:
    cfg = Config(d_model=d, d_ff=FF_RATIO * d)
    runs = run_seeds(lambda s: pretrain(s, cfg))
    WIDTH_RESULTS[d] = np.array([[r["net"][e] for e in ENTITIES] for r in runs])  # seeds x entities
    print(f"d_model={d}: mean net transfer {WIDTH_RESULTS[d].mean():+.2f}  ({time.time() - t0:.0f}s)")
RESULTS["width_sweep"] = {d: v.tolist() for d, v in WIDTH_RESULTS.items()}

means = np.array([WIDTH_RESULTS[d].mean(1) for d in WIDTHS])  # widths x seeds
fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(WIDTHS, means.mean(1), yerr=means.std(1, ddof=1) if len(SEEDS) > 1 else None, marker="o", capsize=3)
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("residual-stream width (d_model)"); ax.set_ylabel("mean net target semantic-logit transfer")
ax.set_title("Subliminal transfer versus residual-stream width")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/residual_width_sweep_mean.pdf"); plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
for j, e in enumerate(ENTITIES):
    ax.plot(WIDTHS, [WIDTH_RESULTS[d][:, j].mean() for d in WIDTHS], marker="o", label=e)
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("residual-stream width (d_model)"); ax.set_ylabel("net target semantic-logit transfer")
ax.set_title("Per-system width sweep"); ax.legend()
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/residual_width_sweep_by_system_appendix.pdf"); plt.show()

## 10. Explicit CAT→7 correlation (appendix)

The paper only gives the figure for this one ("CAT 7 soft-target weight α" against excess generated frequency of 7 and net CAT transfer), so this setup is a reconstruction. Starting from the pretrained checkpoint, we run the prompt-invariant joint training from §7, but CAT-prompt arithmetic targets put mass α on the digit 7. CAT is then genuinely predictive of 7 while the shared-mechanism interference has been trained away. Excess frequency is P(7 | CAT teacher) − P(7 | NEUTRAL teacher).

In [ ]:
SOFT = {}
for alpha in SOFT_ALPHAS:
    runs = run_seeds(lambda s: joint_arithmetic_training(BASE[s], s, varied=True, soft_bias=(7, alpha)),
                     teachers=["NEUTRAL", "CAT"])
    excess = [(r["digit_freqs"]["CAT"][7] - r["digit_freqs"]["NEUTRAL"][7]).item() for r in runs]
    net = [r["net"]["CAT"] for r in runs]
    SOFT[alpha] = {"excess_freq_7": excess, "net_cat_transfer": net}
    print(f"alpha={alpha}: excess freq of 7 {np.mean(excess):+.3f}, net CAT transfer {np.mean(net):+.2f}")
RESULTS["soft_bias"] = SOFT

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, key, ylabel in [(axes[0], "excess_freq_7", "excess generated frequency of 7"),
                        (axes[1], "net_cat_transfer", "net CAT semantic-logit transfer")]:
    vals = np.array([SOFT[a][key] for a in SOFT_ALPHAS])
    ax.errorbar(range(len(SOFT_ALPHAS)), vals.mean(1), yerr=vals.std(1, ddof=1) if len(SEEDS) > 1 else None,
                marker="o", capsize=3)
    ax.set_xticks(range(len(SOFT_ALPHAS)), [f"{a:.2f}" for a in SOFT_ALPHAS])
    ax.set_xlabel("CAT 7 soft-target weight (α)"); ax.set_ylabel(ylabel); ax.axhline(0, color="gray", lw=0.8)
fig.suptitle("Subliminal transfer from an explicit numerical correlation")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/joint_cat_7_soft_bias.pdf"); plt.show()

## Save results

In [ ]:
def to_jsonable(o):
    if isinstance(o, dict):
        return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [to_jsonable(v) for v in o]
    if isinstance(o, torch.Tensor):
        return o.tolist()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, (np.floating, np.integer)):
        return o.item()
    return o


for name, runs in [("main", MAIN), ("sgd", SGD), ("frozen_unembed", FROZEN), ("nongated", NONGATED),
                   ("varied", VARIED), ("drift_matched", DRIFT), ("reversed", REVERSED)]:
    RESULTS[f"{name}_centered"] = [r["centered"] for r in runs]
with open(f"{OUT_DIR}/results.json", "w") as f:
    json.dump(to_jsonable(RESULTS), f, indent=1)
print("saved to", OUT_DIR)